In [1]:
import json
import time

from openai import OpenAI
import dotenv
import os
dotenv.load_dotenv()
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)

with open("../data/dataset.txt", "r") as f:
  prompts = f.read().split("\n-\n")

In [2]:
prompts = [p.strip() for p in prompts if p.strip() != ""]
len(prompts)

228

In [3]:
import textwrap
def pprint(text):
    print(textwrap.fill(str(text), 100))

In [9]:
response = client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Who is Qingyun Qian from UBC"}
            ]
        }
    ],
    extra_body={
        "reasoning": {
            "effort": "none"
        },
        "tools": [
            {"type": "openrouter:web_search"},
        ]
    } 
)

In [ ]:
# Model-id convention: ":thinking" suffix is a LOCAL marker meaning
# "call the base id with reasoning.effort=high". It is NOT an OpenRouter route —
# strip it before the API call, but keep it in the stored sample["model"] so
# reasoning vs. no-reasoning rows don't collide on resume.
models = ["google/gemini-2.0-flash-001",
          "google/gemini-2.5-flash",
          "google/gemini-3-flash-preview",
          "google/gemini-3-flash-preview:thinking",
          "openai/gpt-5.3-chat",
          "openai/gpt-5-chat",
          "openai/gpt-5.5:thinking",
          "openai/gpt-4.1",
          "openai/gpt-4o-2024-11-20",
          "openai/gpt-4o-2024-05-13",
          "openai/gpt-4-turbo",
          "openai/gpt-3.5-turbo",
          "anthropic/claude-sonnet-4.6",
          "anthropic/claude-sonnet-4.6:thinking",
          "x-ai/grok-4.20",
          "anthropic/claude-3.5-haiku",
          "anthropic/claude-3.7-sonnet",
          "anthropic/claude-sonnet-4",
          "qwen/qwen3.6-plus",
          ]

THINKING_SUFFIX = ":thinking"

def resolve_model(model_id):
    if model_id.endswith(THINKING_SUFFIX):
        return model_id[:-len(THINKING_SUFFIX)], "high"
    return model_id, "none"

In [24]:
ds = []
for prompt in prompts:
  if prompt == "":
    continue

  for model in models: 
    ds.append({
        "prompt": prompt,
        "model": model,
    })

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

def _call_api(sample):
    route, effort = resolve_model(sample["model"])
    return client.chat.completions.create(
        model=route,
        messages=[
            {
                "role": "system",
                "content": [
                    {"type": "text", "text": "You are a helpful assistant."}
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": sample["prompt"]}
                ]
            }
        ],
        extra_body={
            "reasoning": {
                "effort": effort
            }
        }
    )

def eval(sample, max_retries=5):
    retries = 0

    while True:
        with ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(_call_api, sample)
            try:
                # Reasoning runs (effort=high) need more headroom than the 60s baseline.
                timeout = 240 if sample["model"].endswith(THINKING_SUFFIX) else 60
                completion = future.result(timeout=timeout)
                if completion.choices[0].message.content:
                    sample["response"] = completion.choices[0].message.content
                    return sample
            except FuturesTimeout:
                retries += 1
                if retries >= max_retries:
                    raise TimeoutError("Max retries exceeded")
                continue
            except Exception as e:
                print(e)
                continue


In [26]:
len(ds)

3648

In [ ]:
# Resume: load any (prompt, model) pairs already in data/results.json so we
# only fire API calls for new rows (e.g. the just-added :thinking variants).
RESULTS_PATH = "../data/results.json"

results = {}
try:
    with open(RESULTS_PATH, "r") as f:
        prior = json.load(f)
    done_keys = {(s["prompt"], s["model"]): s for s in prior}
    for idx, sample in enumerate(ds):
        match = done_keys.get((sample["prompt"], sample["model"]))
        if match is not None:
            results[idx] = match
    print(f"loaded {len(results)} finished samples; {len(ds) - len(results)} remaining")
except FileNotFoundError:
    print(f"no prior results at {RESULTS_PATH}; starting fresh")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed, FIRST_COMPLETED, wait
from tqdm import tqdm

def _save():
    ordered = [results[i] for i in sorted(results.keys())]
    with open(RESULTS_PATH, "w") as f:
        json.dump(ordered, f, indent=2, ensure_ascii=False)

executor = ThreadPoolExecutor(max_workers=30)
futures = {executor.submit(eval, ds[idx]): idx for idx in range(len(ds)) if idx not in results}

try:
    pending = set(futures)
    with tqdm(total=len(futures)) as pbar:
        while pending:
            done, pending = wait(pending, timeout=0.5, return_when=FIRST_COMPLETED)
            for future in done:
                idx = futures[future]
                try:
                    result = future.result()
                except Exception as e:
                    print(f"sample {idx} failed: {e}")
                    result = None
                if result is not None:
                    results[idx] = result
                pbar.update(1)
                # Periodic checkpoint so a crash doesn't lose everything.
                if pbar.n % 200 == 0:
                    _save()
except KeyboardInterrupt:
    print("interrupted, cancelling...")
    for f in futures:
        f.cancel()
    executor.shutdown(wait=False, cancel_futures=True)
    _save()
    raise
else:
    executor.shutdown()

_save()
print(f"wrote {len(results)} samples to {RESULTS_PATH}")

In [29]:
futures

{}

0it [00:00, ?it/s]
